# `evm-intervene`: Counterfactual Intervention Benchmark & GPU Scaling Sweep

**Physical Reasoning Working Group — Answering [`physicalreasoning/forward` Issue #5](https://github.com/physicalreasoning/forward/issues/5)**

This notebook runs the complete **`evm-intervene`** pipeline on a single Google Colab **T4 or L4 GPU** in **~5 minutes** (`< 0.5` Colab compute units):
1. **Zero-Auth Mainnet Data Fetch (`~35s`):** Downloads `31,588` real Ethereum Mainnet Uniswap v3 (`USDC/WETH 0.05%`) swaps, `180` historical `TickLens` concentrated-liquidity boundary maps, and `185` real Aave v3 near-threshold borrower positions from the August 5, 2024 Yen-Carry Crash.
2. **Gate G0 — Exact Simulator Verification (`~8s`):** Verifies the deterministic two-stage counterfactual simulator $\mathcal{F} = \mathcal{F}_{\text{cas}} \circ \mathcal{F}_{\text{AMM}}$ against `31,283` real Mainnet swaps (`4.44e-12 bps` median error).
3. **Mainnet Intervention Benchmark (`~70s`):** Evaluates 4 world-model families (`mlp_direct`, `seq_transformer`, `latent_jepa`, `conservation_wm`) + 5 baselines across 5 seeds on **Regime I (In-Tick)**, **Regime II (Tick-Cliff Crossing)**, and **Regime III (Aave v3 Liquidation Cascade)**.
4. **GPU Scaling Law Sweep (`150k` $\to$ `25M` parameters + `TabPFN v2`, `~3 min`):** Tests whether scaling unstructured neural world models by `160x` in parameters (`150k` $\to$ `25M` in `bf16`/`fp16`) closes the counterfactual intervention gap against the `2,629`-parameter conservation-law world model.

In [ ]:
# 1. Setup Environment, Check GPU & Unpack Clean Repository (~5 seconds)
import base64, io, os, sys, tarfile, time, torch

assert torch.cuda.is_available(), 'Please enable GPU in Colab: Runtime -> Change runtime type -> L4 or T4 GPU'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'[gpu] Detected {gpu_name} ({vram_gb:.1f} GB VRAM)\n')
REPO_TAR_XZ_B64 = '...'
print('[unpack] Decompressing repository and 31,588 Mainnet swaps...')
raw_bytes = base64.b64decode(REPO_TAR_XZ_B64)
with tarfile.open(fileobj=io.BytesIO(raw_bytes), mode='r:xz') as tar:
    members = tar.getmembers()
    for i, member in enumerate(members, 1):
        try:
            tar.extract(member, '.', filter='data')
        except TypeError:
            tar.extract(member, '.')
        pct = (i / len(members)) * 100
        bar = '=' * int(pct // 4) + '-' * (25 - int(pct // 4))
        sys.stdout.write(f'\r[unpack] [{bar}] {pct:5.1f}% ({i}/{len(members)} files): {member.name[:35]:<35}')
        sys.stdout.flush()
print('\n[unpack] Complete! Ready in /content\n')

print('[install] Installing dependencies (transformers, peft, accelerate, bitsandbytes)...')
!pip install -q transformers peft accelerate bitsandbytes scipy tabpfn chronos-forecasting
print('[install] All dependencies ready! Proceed to next cells.\n')


In [ ]:
# 2. Verify Deterministic Simulator on 31,283 Real Mainnet Swaps (Gate G0) (~8 seconds)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, 'scripts/verify_simulator.py'], env=env, check=True)


In [ ]:
# 3. Run Pre-Registered Positive Control (< 15 seconds)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, 'scripts/run_positive_control.py'], env=env, check=True)


In [ ]:
# 4. Research Experiment 1: Ablation Study of the Conservation Law (~2 mins)
# Dissects which structural component governs cliff generalization:
# (none, no_bin_net, uniform_profile, no_liq_coupling)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, '-u', 'scripts/run_research_v2.py', 'ablation', '--epochs', '45'], env=env, check=True)


In [ ]:
# 5. Research Experiment 2: Training Support Coverage Sweep (~4 mins)
# Injects 0%, 1%, 5%, 15% out-of-support simulated large swaps into observational training.
# Tests whether neural world models (MLP, Transformer) fail from architecture or sample coverage.
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, '-u', 'scripts/run_research_v2.py', 'coverage', '--epochs', '45', '--fractions', '0.0', '0.01', '0.05', '0.15'], env=env, check=True)


In [ ]:
# 6. Research Experiment 3: Pretrained LLM World Model Scaling (QLoRA with Tokenized Bins) (~6-10 mins)
# Tokenizes the spatial liquidity manifold into LLM tokens and fine-tunes with LoRA adapters
# across Qwen2.5-0.5B, 1.5B, 3B, and 7B (in 4-bit NF4) plus an untrained random initialization control.
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run(
    [sys.executable, '-u', 'scripts/run_research_v2.py', 'llm', '--models', 'Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-1.5B', 'Qwen/Qwen2.5-3B', 'Qwen/Qwen2.5-7B', '--random-init', 'Qwen/Qwen2.5-0.5B', '--llm-epochs', '4'],
    env=env,
    check=True,
)


In [ ]:
# 7. Generate Summary Tables & Review Research Findings
import json, os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': '.:src'}
subprocess.run([sys.executable, 'scripts/make_report_tables.py'], env=env, check=True)

for exp in ['ablation', 'coverage', 'llm']:
    p = f'results/research_v2/{exp}.json'
    if os.path.exists(p):
        print(f'\n=== {exp.upper()} RESULTS ===')
        data = json.load(open(p))
        print(json.dumps(data, indent=2)[:1200] + '...\n')
